# Measuring overlap of germline and somatic variants by identity

In [1]:
# 1) import modules
import os
import re
import json
import pandas as pd
import scipy
import time
import requests
import hashlib
import csv
import random
from collections import defaultdict
import numpy as np    
import statsmodels.api as sm   
import seaborn as sns
from scipy.signal import find_peaks
import matplotlib.pyplot as plt

In [132]:
start_time_block2=time.asctime(time.localtime())
print(start_time_block2)

cbio_u_all=clinvar_u_all=cbio_no_caid_all=0
#clinvar_missingCA_all_2=pd.DataFrame()
cbio_concat=pd.DataFrame()
clinvar_concat=pd.DataFrame()
overlap=0
overlap_df=pd.DataFrame()

parentpath="USER INSERT PARENT PATH TO WHERE INDIVIDUAL GENE FOLDERS ARE STORED"
os.chdir(parentpath)

#read file with folder names and MANE Select transcript identifiers (or MANE Plus Clinical in the case of SMARCA4)
TranscriptID= pd.read_csv("USER INPUT FILE NAME CONTAINING GENE SYMBOL+NCBI GENE ID+ MANE TRANSCRIPT ID (Supplemental Table S4)", sep="\t")

Hypermutators=pd.read_csv("USER INPUT FILE WITH SAMPLE IDS IN CBIOPORTAL HAVING TUMOR MUTATION BURDEN GREATER THAN 10", sep="\t")


for index, row in TranscriptID.iterrows():
    
    # Loop through each folder name
    folder_name=TranscriptID.loc[index]["Gene Symbol"]
    #if folder name exists, list all files in that folder then change directory to that folder
    if os.path.exists(folder_name) and os.path.isdir(folder_name):# and (folder_name=="CDKN2A"):
        filelist=os.listdir(folder_name)
        os.chdir(folder_name)
        aalength=TranscriptID.loc[index]["aa length"]
        
        print("Current directory:", os.getcwd())
        #load processed raw data cbio:
        cbio_VEP_cancertype=pd.read_csv("USER INPUT PROCESSED SOMATIC DATA FILE NAME", sep="\t")
        
        HGVScremove=["-"]
        cbio_VEP_cancertype=cbio_VEP_cancertype[~cbio_VEP_cancertype["HGVSc"].isin(HGVScremove)==True]
        
        
        VEP_categories_tofilter_list=["synonymous_variant","5_prime_UTR_variant", "3_prime_UTR_variant", "non_coding_transcript_exon_variant", "intron_variant", "non_coding_transcript_variant", "upstream_gene_variant", "downstream_gene_variant", "regulatory_region_ablation", "regulatory_region_amplification", "regulatory_region_variant", "intergenic_variant", "splice_region_variant"]
        cbio_VEP_cancertype=cbio_VEP_cancertype[~cbio_VEP_cancertype["collapsed_Consequence"].isin(VEP_categories_tofilter_list)]
        
        Hypermutators_to_filter=Hypermutators.set_index('uniqueSampleKey').index
        #print("with hypermutators count")
        totalbeforehypermutatorfilter=len(cbio_VEP_cancertype)
        #print(len(cbio_VEP_cancertype))
        cbio_VEP_cancertype=cbio_VEP_cancertype[~cbio_VEP_cancertype["uniqueSampleKey"].isin(Hypermutators_to_filter)]
        
        cbio_VEP=cbio_VEP_cancertype
        
        # filter out the variants that have 0 reads of the tumor allele (likely invalid entries)
        cbio_rawdata_s_u_0tumorAltCount=cbio_VEP[cbio_VEP["tumorAltCount"]==0]
        cbio_rawdata_s_u_no0tumorAltCount=cbio_VEP.drop(cbio_rawdata_s_u_0tumorAltCount.index)
        #print(len(cbio_rawdata_s_u_no0tumorAltCount))
        Four_no0tumorAltCount=len(cbio_rawdata_s_u_no0tumorAltCount)
        cbio_VEP=cbio_rawdata_s_u_no0tumorAltCount
        
        
        clinvar_cbio_tocompare=pd.read_csv("USER INPUT FILE NAME OF SOMATIC VARIANTS IN CLINVAR UNFILTERED FOR PATHOGENICITY", sep="\t")
        Variation_interpretation_toexclude = ["Pathogenic", "Likely pathogenic", "Pathogenic/Likely pathogenic", "Conflicting interpretations of pathogenicity greater than or equal to 75", "Pathogenic/Likely pathogenic/Pathogenic, low penetrance/Established risk allele"]
        clinvar_cbio_tocompare_VUSLBB=clinvar_cbio_tocompare[~clinvar_cbio_tocompare["GL_first_Description"].isin(Variation_interpretation_toexclude)]
        
        cbio_VEP_cancertype=cbio_VEP
        cbio_u_df=cbio_VEP_cancertype.drop_duplicates(subset=["level_0"])
        cbio_concat=pd.concat([cbio_concat,cbio_u_df])
       
        cbio_nomissingCA=cbio_VEP_cancertype[~cbio_VEP_cancertype["@id"].astype(str).str.contains("_:CA")]
        cbio_missingCA=cbio_VEP_cancertype[cbio_VEP_cancertype["@id"].astype(str).str.contains("_:CA")]
        cbio_VEP_cancertype=cbio_nomissingCA[cbio_nomissingCA["@id"].isnull()==False]
        cbio_VEP_cancertype_u=cbio_VEP_cancertype.drop_duplicates(subset=["level_0"])
        
        cbio_u=len(cbio_VEP_cancertype.drop_duplicates(subset=["level_0"]))
        cbio_u_all=cbio_u_all+cbio_u

        cbio_no_caid=len(cbio_missingCA.drop_duplicates(subset=["level_0"]))
        cbio_no_caid_all=cbio_no_caid_all+cbio_no_caid

         
        
        clinvar_VEP_cancertype=pd.read_csv("USER INPUT PROCESSED GERMLINE DATA FILE NAME", sep="\t")
        clinvar_VEP_cancertype=clinvar_VEP_cancertype[~clinvar_VEP_cancertype["collapsed_Consequence"].isin(VEP_categories_tofilter_list)]
        clinvar_VEP_cancertype_backup=clinvar_VEP_cancertype
        
        clinvar_nomissingCA=clinvar_VEP_cancertype[~clinvar_VEP_cancertype["@id"].astype(str).str.contains("_:CA")]
        clinvar_VEP_cancertype=clinvar_nomissingCA[clinvar_nomissingCA["@id"].isnull()==False]

        
        #clinvar_missingCA=clinvar_VEP_cancertype_backup[clinvar_VEP_cancertype_backup["@id"].astype(str).str.contains("_:CA")]
        #clinvar_missingCA=clinvar_VEP_cancertype_backup[clinvar_VEP_cancertype_backup["@id"].isnull()==True]
        #display(clinvar_missingCA)
        #clinvar_missingCA=clinvar_VEP_cancertype_backup.drop(index=clinvar_nomissingCA.index)
        #print(len(clinvar_missingCA))
        #clinvar_missingCA_all_2=pd.concat([clinvar_missingCA_all_2,clinvar_missingCA])
        ##cbio_missingCA_unique=cbio_missingCA.drop_duplicates(subset=["level_0"])
        #hgvsc_compare=clinvar_missingCA.set_index("HGVSc").join(cbio_missingCA_unique.set_index("HGVSc"), how="inner", rsuffix="_cbio")
        #display(hgvsc_compare)
                                                             
        
        clinvar_u=len(clinvar_VEP_cancertype)
        clinvar_u_all=clinvar_u_all+clinvar_u
        
        clinvar_concat=pd.concat([clinvar_concat,clinvar_VEP_cancertype_backup])

        join=clinvar_VEP_cancertype.set_index("@id").join(cbio_VEP_cancertype_u.set_index("@id"), how="inner", rsuffix="_cbio")
        overlap=overlap+len(join)
        overlap_df=pd.concat([overlap_df, join])
    
 
        print("Current directory:", os.getcwd())
        os.chdir(parentpath)
    else:
        print(f"Folder '{folder_name}' not found.")

end_time_block2=time.asctime(time.localtime())
print(end_time_block2)

Tue Sep  2 23:45:10 2025
Current directory: /Users/suhasinilulla/Library/CloudStorage/Box-Box/PhD_BCM/Lulla_Plon_Lab/High_priority_gene_list/GL_S_Pipeline_1st_pass_47TSGs_May2023/WT1
Current directory: /Users/suhasinilulla/Library/CloudStorage/Box-Box/PhD_BCM/Lulla_Plon_Lab/High_priority_gene_list/GL_S_Pipeline_1st_pass_47TSGs_May2023/WT1
Current directory: /Users/suhasinilulla/Library/CloudStorage/Box-Box/PhD_BCM/Lulla_Plon_Lab/High_priority_gene_list/GL_S_Pipeline_1st_pass_47TSGs_May2023/VHL
Current directory: /Users/suhasinilulla/Library/CloudStorage/Box-Box/PhD_BCM/Lulla_Plon_Lab/High_priority_gene_list/GL_S_Pipeline_1st_pass_47TSGs_May2023/VHL
Current directory: /Users/suhasinilulla/Library/CloudStorage/Box-Box/PhD_BCM/Lulla_Plon_Lab/High_priority_gene_list/GL_S_Pipeline_1st_pass_47TSGs_May2023/TSC2
Current directory: /Users/suhasinilulla/Library/CloudStorage/Box-Box/PhD_BCM/Lulla_Plon_Lab/High_priority_gene_list/GL_S_Pipeline_1st_pass_47TSGs_May2023/TSC2
Current directory: /Users

/var/folders/84/djb4q50s33b6t8_cvx6h9src0000gn/T/ipykernel_62623/3911374955.py:32: DtypeWarning: Columns (44,48) have mixed types. Specify dtype option on import or set low_memory=False.
  cbio_VEP_cancertype=pd.read_csv("Python_coderun_7-11-24_cbio_process_again/Python_cbio_processed_VEP_MANE_CAID_8-16-24.txt", sep="\t")


Current directory: /Users/suhasinilulla/Library/CloudStorage/Box-Box/PhD_BCM/Lulla_Plon_Lab/High_priority_gene_list/GL_S_Pipeline_1st_pass_47TSGs_May2023/TP53
Current directory: /Users/suhasinilulla/Library/CloudStorage/Box-Box/PhD_BCM/Lulla_Plon_Lab/High_priority_gene_list/GL_S_Pipeline_1st_pass_47TSGs_May2023/SUFU
Current directory: /Users/suhasinilulla/Library/CloudStorage/Box-Box/PhD_BCM/Lulla_Plon_Lab/High_priority_gene_list/GL_S_Pipeline_1st_pass_47TSGs_May2023/SUFU
Current directory: /Users/suhasinilulla/Library/CloudStorage/Box-Box/PhD_BCM/Lulla_Plon_Lab/High_priority_gene_list/GL_S_Pipeline_1st_pass_47TSGs_May2023/STK11
Current directory: /Users/suhasinilulla/Library/CloudStorage/Box-Box/PhD_BCM/Lulla_Plon_Lab/High_priority_gene_list/GL_S_Pipeline_1st_pass_47TSGs_May2023/STK11
Current directory: /Users/suhasinilulla/Library/CloudStorage/Box-Box/PhD_BCM/Lulla_Plon_Lab/High_priority_gene_list/GL_S_Pipeline_1st_pass_47TSGs_May2023/SMARCB1
Current directory: /Users/suhasinilulla/L

In [167]:
clinvar_concat_minus_overlap=clinvar_concat.set_index("@id").drop(index=overlap_df.index)

In [170]:
all_concat=pd.concat([clinvar_concat_minus_overlap.reset_index(), cbio_concat])

In [176]:
all_concat["collapsed_Consequence"].value_counts()

collapsed_Consequence
frameshift_variant          24543
stop_gained                  7536
missense_variant             3708
splice_donor_variant         2487
splice_acceptor_variant      2309
>=50bp_indel                  650
inframe_deletion              414
inframe_insertion             143
start_lost                    122
protein_altering_variant       37
stop_lost                      35
coding_sequence_variant         1
Name: count, dtype: int64

In [467]:
all_concat["collapsed_Consequence"].value_counts().sum()

41985

In [ ]:
overlap_df["collapsed_Consequence"].value_counts().sum() ## =3863 (Figure 2 panel A)

In [177]:
overlap_df["collapsed_Consequence"].value_counts() #Figure 2 panel D

collapsed_Consequence
stop_gained                1446
frameshift_variant         1121
missense_variant            553
splice_donor_variant        364
splice_acceptor_variant     349
inframe_deletion             18
start_lost                    9
stop_lost                     2
>=50bp_indel                  1
Name: count, dtype: int64

In [895]:
clinvar_concat["collapsed_Consequence"].value_counts() #Figure 2 panel D

collapsed_Consequence
frameshift_variant          19218
stop_gained                  6295
missense_variant             2674
splice_donor_variant         2027
splice_acceptor_variant      1767
>=50bp_indel                  575
inframe_deletion              200
start_lost                     95
inframe_insertion              59
protein_altering_variant       16
stop_lost                      14
coding_sequence_variant         1
Name: count, dtype: int64

In [897]:
cbio_concat["collapsed_Consequence"].value_counts() #Figure 2 panel D

collapsed_Consequence
frameshift_variant          6446
stop_gained                 2687
missense_variant            1587
splice_acceptor_variant      891
splice_donor_variant         824
inframe_deletion             232
inframe_insertion             84
>=50bp_indel                  76
start_lost                    36
stop_lost                     23
protein_altering_variant      21
Name: count, dtype: int64

In [178]:
clinvar_concat["Occurrence"].value_counts() # used for figure S3 panel A

Occurrence
1     18232
2      6416
3      3023
4      1761
5      1049
6       634
7       401
8       305
9       226
10      183
11      121
13       94
12       92
14       57
15       56
16       44
17       39
18       30
20       26
23       19
19       19
22       15
21       13
27       11
24       11
30       10
25        7
31        6
32        4
28        4
38        4
40        4
26        3
41        3
60        2
37        2
39        2
33        2
34        2
29        2
50        1
43        1
35        1
44        1
48        1
63        1
65        1
Name: count, dtype: int64

In [3]:
start_time_block2=time.asctime(time.localtime())
print(start_time_block2)

sharedvarcounts=pd.DataFrame()


parentpath="USER INSERT PARENT PATH TO WHERE INDIVIDUAL GENE FOLDERS ARE STORED"
os.chdir(parentpath)

Hypermutators=pd.read_csv("USER INPUT FILE WITH SAMPLE IDS IN CBIOPORTAL HAVING TUMOR MUTATION BURDEN GREATER THAN 10", sep="\t")
Hypermutators_to_filter=Hypermutators.set_index('uniqueSampleKey').index

#read file with folder names and MANE Select transcript identifiers (or MANE Plus Clinical in the case of SMARCA4)
TranscriptID= pd.read_csv("USER INPUT FILE NAME CONTAINING GENE SYMBOL+NCBI GENE ID+ MANE TRANSCRIPT ID (Supplemental Table S4)", sep="\t")

for index, row in TranscriptID.iterrows():
    
    # Loop through each folder name
    folder_name=TranscriptID.loc[index]["Gene Symbol"]
    #if folder name exists, list all files in that folder then change directory to that folder
    if os.path.exists(folder_name) and os.path.isdir(folder_name):
        filelist=os.listdir(folder_name)
        os.chdir(folder_name)
        
        clinvar=pd.read_csv("USER INPUT PROCESSED GERMLINE DATA FILE NAME", sep="\t")
        
        
        cbio_CAID=pd.read_csv("SER INPUT PROCESSED SOMATIC DATA FILE NAME", sep="\t")
        cbio_CAID=cbio_CAID[~cbio_CAID["uniqueSampleKey"].isin(Hypermutators_to_filter)]
        VEP_categories_tofilter_list=["synonymous_variant","5_prime_UTR_variant", "3_prime_UTR_variant", "non_coding_transcript_exon_variant", "intron_variant", "non_coding_transcript_variant", "upstream_gene_variant", "downstream_gene_variant", "regulatory_region_ablation", "regulatory_region_amplification", "regulatory_region_variant", "intergenic_variant", "splice_region_variant"]
        cbio_CAID=cbio_CAID[~cbio_CAID["collapsed_Consequence"].isin(VEP_categories_tofilter_list)]
        clinvar=clinvar[~clinvar["collapsed_Consequence"].isin(VEP_categories_tofilter_list)]
        
        cbio_CAID=cbio_CAID.drop_duplicates(subset=['level_0'])
        
        clinvar_nomissingID=clinvar[~clinvar["HGVSp"].astype(str).isin(["-"])]
        clinvar_nomissingID=clinvar_nomissingID.drop_duplicates(subset=["HGVSp"])
        cbio_CAID_nomissingID=cbio_CAID[~cbio_CAID["HGVSp"].astype(str).isin(["-"])]
        cbio_CAID_nomissingID=cbio_CAID_nomissingID.drop_duplicates(subset=["HGVSp"])
        
        sharedvars= clinvar_nomissingID.set_index("HGVSp").join(cbio_CAID_nomissingID.set_index("HGVSp"), how="inner", lsuffix="clinvar", rsuffix="cbio")
        #print(len(sharedvars))
        
        outdf1=pd.DataFrame({'Gene':[folder_name],'clinvarcount':[len(clinvar)],'clinvarcount_hashgvsp':[len(clinvar_nomissingID)],'cbiocount':[len(cbio_CAID)], 'cbiocount_hashgvsp':[len(cbio_CAID_nomissingID)],'sharedvars':[len(sharedvars)]})
        sharedvarcounts=pd.concat([sharedvarcounts,outdf1])
        
        
        #print("Current directory:", os.getcwd())
        os.chdir(parentpath)
    else:
        print(f"Folder '{folder_name}' not found.")

display(sharedvarcounts)
end_time_block2=time.asctime(time.localtime())
print(end_time_block2)



Tue Feb 17 12:14:54 2026


/var/folders/84/djb4q50s33b6t8_cvx6h9src0000gn/T/ipykernel_48100/1664852398.py:28: DtypeWarning: Columns (44,48) have mixed types. Specify dtype option on import or set low_memory=False.
  cbio_CAID=pd.read_csv("Python_coderun_7-11-24_cbio_process_again/Python_cbio_processed_VEP_MANE_CAID_8-16-24.txt", sep="\t")


,Gene,clinvarcount,clinvarcount_hashgvsp,cbiocount,cbiocount_hashgvsp,sharedvars
0,WT1,92,80,93,76,9
0,VHL,348,277,393,326,97
0,TSC2,1083,831,248,188,64
0,TSC1,576,460,209,164,63
0,TP53,823,679,2620,2182,485
0,SUFU,84,63,55,43,9
0,STK11,274,208,327,251,67
0,SMARCB1,71,50,71,61,14
0,SMARCA4,235,190,268,212,29
0,SMAD4,227,191,555,456,74


Tue Feb 17 12:14:57 2026


In [4]:
sharedvarcounts.sum() #Figure 2 Panel C

Gene                     WT1VHLTSC2TSC1TP53SUFUSTK11SMARCB1SMARCA4SMAD4...
clinvarcount                                                         32941
clinvarcount_hashgvsp                                                26614
cbiocount                                                            12954
cbiocount_hashgvsp                                                   10535
sharedvars                                                            3274
dtype: object